# RSNN on FI-2010 — V3b: Tuning BNTT+LT

**V3 result**: BNTT + Learnable Threshold reached 59.15% (from 37.56% base). Gap to LSTM: 6.8pp.

**This notebook tests whether we can close the gap further without changing the RSNN architecture.**

All experiments use BNTT+LT as the base. The RSNN architecture (LIF neurons, surrogate gradient, recurrent connections, max-over-time readout) is identical to Cramer et al. (2020).

| Experiment | What it changes |
|---|---|
| T1: 2-layer BNTT+LT | Adds depth (worked on ECG: +2.8pp) |
| T2: Tau sweep on BNTT+LT | Finds optimal time constant for LOB data |
| T3: Learning rate sweep | Tests whether 1e-3 is optimal for BNTT+LT |
| T4: Hidden size sweep with BNTT | Tests 128/256/512 now that neurons actually fire |
| T5: Best combo | Combines best tau + best lr + best depth |

**Controls**: Same V1 balanced test set (139,488 samples), same Trainer, same class-weighted CE.

**V3 baselines**:
- LSTM: 65.99%
- CNN: 65.47%
- RSNN base (th=1.0): 37.56%
- RSNN BNTT+LT: 59.15%


## 0. Setup

In [1]:
import os, glob, subprocess, time, json, copy
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, f1_score
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

print("Setup complete")


Device: cuda
GPU: Tesla T4
Setup complete


## 1. Load FI-2010 (V1 protocol, all 3 test files)

In [2]:
print("Downloading FI-2010...")
subprocess.run(['wget', '-q',
    'https://raw.githubusercontent.com/zcakhaa/DeepLOB-Deep-Convolutional-Neural-Networks-for-Limit-Order-Books/master/data/data.zip',
    '-O', '/kaggle/working/data.zip'], check=True)
subprocess.run(['unzip', '-q', '-o', '/kaggle/working/data.zip', '-d', '/kaggle/working/'], check=True)
DATA_DIR = '/kaggle/working'
print("Done.")

test_files = sorted(glob.glob(os.path.join(DATA_DIR, "Test_Dst_NoAuction*.txt")))
print(f"Test files: {[os.path.basename(f) for f in test_files]}")

def prepare_x(data):
    return data[:40, :].T.astype(np.float32)

def get_label(data):
    return data[-5:, :].T.astype(int) - 1

def data_classification(X, Y, T=100):
    N = X.shape[0]
    samples = N - T + 1
    X_seq = np.zeros((samples, T, X.shape[1]), dtype=np.float32)
    for i in range(samples):
        X_seq[i] = X[i:i+T]
    return X_seq, Y[T-1:]

dec_train = np.loadtxt(os.path.join(DATA_DIR, 'Train_Dst_NoAuction_DecPre_CF_7.txt'))
dec_test = np.hstack([np.loadtxt(tf) for tf in test_files])

train_lob, train_label = prepare_x(dec_train), get_label(dec_train)
test_lob, test_label = prepare_x(dec_test), get_label(dec_test)

T = 100
HORIZON = 3

X_train_seq, y_train_seq = data_classification(train_lob, train_label, T=T)
X_test_seq, y_test_seq = data_classification(test_lob, test_label, T=T)

y_train_all = y_train_seq[:, HORIZON]
y_test = y_test_seq[:, HORIZON]

val_split = int(len(X_train_seq) * 0.8)
X_val = X_train_seq[val_split:]
y_val = y_train_all[val_split:]
X_train = X_train_seq[:val_split]
y_train = y_train_all[:val_split]

print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test_seq.shape}")
for i, name in enumerate(['Down', 'Stationary', 'Up']):
    n = (y_test == i).sum()
    print(f"  {name}: {n} ({n/len(y_test)*100:.1f}%)")


Done.
Test files: ['Test_Dst_NoAuction_DecPre_CF_7.txt', 'Test_Dst_NoAuction_DecPre_CF_8.txt', 'Test_Dst_NoAuction_DecPre_CF_9.txt']


Train: (203720, 100, 40) | Val: (50931, 100, 40) | Test: (139488, 100, 40)
  Down: 38408 (27.5%)
  Stationary: 65996 (47.3%)
  Up: 35084 (25.2%)


## 2. Dataset, Components, Trainer

In [3]:
class LOBDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

BS = 256
train_ds = LOBDataset(X_train, y_train)
val_ds = LOBDataset(X_val, y_val)
test_ds = LOBDataset(X_test_seq, y_test)
train_ld = DataLoader(train_ds, BS, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
val_ld = DataLoader(val_ds, BS, shuffle=False, num_workers=2, pin_memory=True)
test_ld = DataLoader(test_ds, BS, shuffle=False, num_workers=2, pin_memory=True)

counts = torch.bincount(torch.tensor(y_train), minlength=3).float()
class_weights = ((1.0 / counts) / (1.0 / counts).sum() * 3).to(device)
print(f"Class weights: {class_weights.cpu().numpy()}")


Class weights: [0.9467207 1.0937424 0.9595369]


In [4]:
class SurrogateSpike(torch.autograd.Function):
    beta = 40.0
    @staticmethod
    def forward(ctx, mem, threshold=1.0):
        ctx.save_for_backward(mem)
        ctx.threshold = threshold
        return (mem >= threshold).float()
    @staticmethod
    def backward(ctx, grad_output):
        mem, = ctx.saved_tensors
        v = mem - ctx.threshold
        grad = 1.0 / (1.0 + SurrogateSpike.beta * torch.abs(v)) ** 2
        return grad_output * grad, None

def spike_fn(x, threshold=1.0):
    return SurrogateSpike.apply(x, threshold)


class ReadoutLayer(nn.Module):
    def __init__(self, input_size, output_size, tau_mem=20.0, dt=10.0):
        super().__init__()
        self.fc = nn.Linear(input_size, output_size, bias=False)
        self.beta = np.exp(-dt / tau_mem)
        nn.init.kaiming_uniform_(self.fc.weight, nonlinearity='linear')
    def forward(self, x):
        B, T_s, _ = x.shape
        mem = torch.zeros(B, self.fc.out_features, device=x.device)
        mem_rec = []
        for t in range(T_s):
            mem = self.beta * mem + (1 - self.beta) * self.fc(x[:, t])
            mem_rec.append(mem)
        return torch.stack(mem_rec, dim=1)


def spike_regularization(all_spikes, theta_l=0.01, s_l=1.0, theta_u=100.0, s_u=0.06):
    reg = torch.tensor(0.0, device=all_spikes[0].device)
    for spk in all_spikes:
        B, T_s, N = spk.shape
        mean_rate = spk.sum(dim=1) / T_s
        reg += s_l / (B * N) * (F.relu(theta_l - mean_rate) ** 2).sum()
        pop_count = spk.sum(dim=(1, 2)) / N
        reg += s_u / B * (F.relu(pop_count - theta_u) ** 2).sum()
    return reg

print("Components loaded")


Components loaded


In [5]:
class LIFLayerBNTT_LT(nn.Module):
    def __init__(self, input_size, hidden_size, n_steps, recurrent=False,
                 tau_mem_init=20.0, tau_syn_init=10.0, dt=10.0,
                 learnable_tau=False, dropout=0.0, threshold_init=0.1):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.recurrent = recurrent
        self.dt = dt
        self.dropout = dropout

        self.W_ff = nn.Linear(input_size, hidden_size, bias=False)
        if recurrent:
            self.W_rec = nn.Linear(hidden_size, hidden_size, bias=False)

        self.bn = nn.ModuleList([nn.BatchNorm1d(hidden_size) for _ in range(n_steps)])
        self.log_threshold = nn.Parameter(torch.tensor(np.log(threshold_init)))

        if learnable_tau:
            self.log_tau_mem = nn.Parameter(torch.tensor(np.log(tau_mem_init)))
            self.log_tau_syn = nn.Parameter(torch.tensor(np.log(tau_syn_init)))
        else:
            self.register_buffer('log_tau_mem', torch.tensor(np.log(tau_mem_init)))
            self.register_buffer('log_tau_syn', torch.tensor(np.log(tau_syn_init)))

        nn.init.kaiming_uniform_(self.W_ff.weight, nonlinearity='linear')
        if recurrent:
            nn.init.kaiming_uniform_(self.W_rec.weight, nonlinearity='linear')

    @property
    def alpha(self):
        return torch.exp(-self.dt / torch.exp(self.log_tau_syn))
    @property
    def beta_decay(self):
        return torch.exp(-self.dt / torch.exp(self.log_tau_mem))
    @property
    def threshold(self):
        return torch.exp(self.log_threshold)

    def forward(self, x):
        B, T_steps, _ = x.shape
        alpha, beta = self.alpha, self.beta_decay
        thr = self.threshold
        syn = torch.zeros(B, self.hidden_size, device=x.device)
        mem = torch.zeros(B, self.hidden_size, device=x.device)
        prev_spk = torch.zeros(B, self.hidden_size, device=x.device)
        spk_rec, mem_rec = [], []
        for t in range(T_steps):
            cur = self.bn[t](self.W_ff(x[:, t]))
            syn = alpha * syn + cur
            if self.recurrent:
                rec_in = prev_spk
                if self.dropout > 0 and self.training:
                    rec_in = F.dropout(rec_in, p=self.dropout)
                syn = syn + self.W_rec(rec_in)
            mem = beta * mem * (1.0 - prev_spk) + (1.0 - beta) * syn
            spk = spike_fn(mem, thr)
            spk_rec.append(spk)
            mem_rec.append(mem)
            prev_spk = spk
        return torch.stack(spk_rec, dim=1), torch.stack(mem_rec, dim=1)


class SNN_BNTT_LT(nn.Module):
    def __init__(self, input_size=40, hidden_size=256, output_size=3, n_steps=100,
                 n_hidden_layers=1, recurrent=True, tau_mem=20.0, tau_syn=10.0,
                 dt=10.0, learnable_tau=True, dropout=0.3, threshold_init=0.1):
        super().__init__()
        self.hidden_layers = nn.ModuleList()
        in_sz = input_size
        for _ in range(n_hidden_layers):
            self.hidden_layers.append(
                LIFLayerBNTT_LT(in_sz, hidden_size, n_steps, recurrent=recurrent,
                                tau_mem_init=tau_mem, tau_syn_init=tau_syn, dt=dt,
                                learnable_tau=learnable_tau, dropout=dropout,
                                threshold_init=threshold_init))
            in_sz = hidden_size
        self.readout = ReadoutLayer(hidden_size, output_size, tau_mem=tau_mem, dt=dt)

    def forward(self, x):
        all_spikes = []
        for layer in self.hidden_layers:
            spikes, _ = layer(x)
            all_spikes.append(spikes)
            x = spikes
        out_mem = self.readout(spikes)
        output, _ = torch.max(out_mem, dim=1)
        return output, all_spikes, out_mem

    def count_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def get_thresholds(self):
        return [layer.threshold.item() for layer in self.hidden_layers]

print("BNTT+LT architecture loaded (supports multi-layer)")
m = SNN_BNTT_LT(40, 256, 3, n_steps=100, n_hidden_layers=1)
print(f"1-layer: {m.count_params():,} params")
del m
m = SNN_BNTT_LT(40, 256, 3, n_steps=100, n_hidden_layers=2)
print(f"2-layer: {m.count_params():,} params")
del m


BNTT+LT architecture loaded (supports multi-layer)
1-layer: 127,747 params
2-layer: 310,022 params


In [6]:
class Trainer:
    def __init__(self, model, train_ld, val_ld, test_ld, lr=1e-3, device='cuda'):
        self.model = model.to(device)
        self.train_ld, self.val_ld, self.test_ld = train_ld, val_ld, test_ld
        self.device = device
        self.optimizer = torch.optim.Adamax(model.parameters(), lr=lr)
        self.criterion = nn.CrossEntropyLoss(weight=class_weights)
        self.history = {'train_acc': [], 'val_acc': [], 'spike_rates': [], 'epoch_time': []}

    def _run_epoch(self, loader, train=False):
        self.model.train() if train else self.model.eval()
        correct, total, spk_rates = 0, 0, []
        ctx = torch.enable_grad() if train else torch.no_grad()
        with ctx:
            for x, y in loader:
                x, y = x.to(self.device), y.to(self.device)
                logits, all_spk, _ = self.model(x)
                loss = self.criterion(logits, y)
                if train:
                    loss = loss + spike_regularization(all_spk)
                    self.optimizer.zero_grad()
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                    self.optimizer.step()
                correct += (logits.argmax(1) == y).sum().item()
                total += x.size(0)
                spk_rates.extend([s.detach().mean().item() for s in all_spk])
        return correct / total, np.mean(spk_rates)

    def train(self, n_epochs=80, patience=20, verbose=True):
        best_val, best_state, no_improve = 0, None, 0
        for ep in range(n_epochs):
            t0 = time.time()
            tr_a, sr = self._run_epoch(self.train_ld, train=True)
            va_a, _ = self._run_epoch(self.val_ld)
            dt_ep = time.time() - t0
            self.history['train_acc'].append(tr_a)
            self.history['val_acc'].append(va_a)
            self.history['spike_rates'].append(sr)
            self.history['epoch_time'].append(dt_ep)
            if va_a > best_val + 0.001:
                best_val = va_a
                no_improve = 0
                best_state = {k: v.cpu().clone() for k, v in self.model.state_dict().items()}
            else:
                no_improve += 1
            if verbose and (ep % 5 == 0 or no_improve >= patience):
                star = ' *' if no_improve == 0 else ''
                print(f"Ep {ep:3d} | Tr: {tr_a:.4f} | Va: {va_a:.4f} | Spk: {sr:.4f} | Best: {best_val:.4f} | {dt_ep:.0f}s{star}")
            if no_improve >= patience:
                print(f"Early stop at epoch {ep}")
                break
        if best_state:
            self.model.load_state_dict({k: v.to(self.device) for k, v in best_state.items()})
        te_a, _ = self._run_epoch(self.test_ld)
        mins = sum(self.history['epoch_time']) / 60
        print(f"Best val: {best_val*100:.2f}% | Test: {te_a*100:.2f}% | Time: {mins:.0f}min")
        return te_a

print("Trainer ready")


Trainer ready


## 3. Experiment T1: 2-Layer BNTT+LT

On ECG, 2 layers improved from 62.49% to 65.26% (+2.8pp).
The LOB has hierarchical structure (price levels, bid/ask sides) that a second layer might capture.

In [7]:
RESULTS = {'BNTT_LT_1L_base': 0.5915}  # V3 result carried forward

print("=" * 60)
print("T1: 2-LAYER BNTT+LT")
print("=" * 60)

model_2l = SNN_BNTT_LT(40, 256, 3, n_steps=T, n_hidden_layers=2,
                        tau_mem=20.0, tau_syn=10.0, threshold_init=0.1)
print(f"Params: {model_2l.count_params():,}")
trainer_2l = Trainer(model_2l, train_ld, val_ld, test_ld, lr=1e-3, device=device)
acc_2l = trainer_2l.train(n_epochs=80, patience=20)
RESULTS['T1_2layer'] = acc_2l
print(f"Learned thresholds: {model_2l.get_thresholds()}")


T1: 2-LAYER BNTT+LT
Params: 310,022


Ep   0 | Tr: 0.3821 | Va: 0.3700 | Spk: 0.4378 | Best: 0.3700 | 263s *


Ep   5 | Tr: 0.4004 | Va: 0.3296 | Spk: 0.4543 | Best: 0.3700 | 260s


Ep  10 | Tr: 0.4588 | Va: 0.3700 | Spk: 0.4745 | Best: 0.3700 | 266s


Ep  15 | Tr: 0.5015 | Va: 0.3700 | Spk: 0.5011 | Best: 0.3700 | 259s


Ep  20 | Tr: 0.5288 | Va: 0.3700 | Spk: 0.5012 | Best: 0.3700 | 267s
Early stop at epoch 20


Best val: 37.00% | Test: 42.26% | Time: 92min
Learned thresholds: [0.10000000000000002, 0.10000000000000002]


## 4. Experiment T2: Tau Sweep on BNTT+LT

On ECG, tau=80ms was best (vs default 20ms). LOB events are fast (~100ms apart),
so shorter tau might be better. But now that BNTT rescales inputs, the optimal tau may shift.

In [8]:
print("=" * 60)
print("T2: TAU SWEEP (1-layer BNTT+LT)")
print("=" * 60)

tau_results = {}
for tau in [5.0, 10.0, 40.0, 80.0]:
    print(f"--- tau_mem={tau}ms ---")
    m = SNN_BNTT_LT(40, 256, 3, n_steps=T, n_hidden_layers=1,
                     tau_mem=tau, tau_syn=tau/2, threshold_init=0.1)
    t = Trainer(m, train_ld, val_ld, test_ld, lr=1e-3, device=device)
    acc = t.train(n_epochs=80, patience=20, verbose=False)
    tau_results[tau] = acc
    RESULTS[f'T2_tau{int(tau)}'] = acc
    print(f"  tau={tau}ms -> {acc*100:.2f}%")
    print()

best_tau = max(tau_results, key=tau_results.get)
print(f"Best tau: {best_tau}ms -> {tau_results[best_tau]*100:.2f}%")
print(f"(Base tau=20ms -> 59.15%)")


T2: TAU SWEEP (1-layer BNTT+LT)
--- tau_mem=5.0ms ---


Early stop at epoch 20


Best val: 37.00% | Test: 42.26% | Time: 52min
  tau=5.0ms -> 42.26%

--- tau_mem=10.0ms ---


Early stop at epoch 20


Best val: 36.99% | Test: 43.23% | Time: 52min
  tau=10.0ms -> 43.23%

--- tau_mem=40.0ms ---


Early stop at epoch 79


Best val: 46.60% | Test: 58.57% | Time: 203min
  tau=40.0ms -> 58.57%

--- tau_mem=80.0ms ---


Early stop at epoch 72


Best val: 41.30% | Test: 49.56% | Time: 192min
  tau=80.0ms -> 49.56%

Best tau: 40.0ms -> 58.57%
(Base tau=20ms -> 59.15%)


## 5. Experiment T3: Learning Rate Sweep

The default lr=1e-3 was inherited from SHD. BNTT adds many more parameters
(100 BN layers), which may benefit from a different learning rate.

In [ ]:
print("=" * 60)
print("T3: LEARNING RATE SWEEP (1-layer BNTT+LT)")
print("=" * 60)

lr_results = {}
for lr in [5e-4, 2e-3, 5e-3]:
    print(f"--- lr={lr} ---")
    m = SNN_BNTT_LT(40, 256, 3, n_steps=T, n_hidden_layers=1,
                     tau_mem=20.0, tau_syn=10.0, threshold_init=0.1)
    t = Trainer(m, train_ld, val_ld, test_ld, lr=lr, device=device)
    acc = t.train(n_epochs=80, patience=20, verbose=False)
    lr_results[lr] = acc
    RESULTS[f'T3_lr{lr}'] = acc
    print(f"  lr={lr} -> {acc*100:.2f}%")
    print()

best_lr = max(lr_results, key=lr_results.get)
print(f"Best lr: {best_lr} -> {lr_results[best_lr]*100:.2f}%")
print(f"(Base lr=1e-3 -> 59.15%)")


## 6. Experiment T4: Hidden Size Sweep with BNTT+LT

V2 tested hidden sizes without BNTT and found no difference (neurons weren't firing).
Now that BNTT activates the spiking dynamics, hidden size should matter.

In [ ]:
print("=" * 60)
print("T4: HIDDEN SIZE SWEEP (1-layer BNTT+LT)")
print("=" * 60)

hs_results = {}
for hs in [128, 512]:
    print(f"--- hidden_size={hs} ---")
    m = SNN_BNTT_LT(40, hs, 3, n_steps=T, n_hidden_layers=1,
                     tau_mem=20.0, tau_syn=10.0, threshold_init=0.1)
    print(f"  Params: {m.count_params():,}")
    t = Trainer(m, train_ld, val_ld, test_ld, lr=1e-3, device=device)
    acc = t.train(n_epochs=80, patience=20, verbose=False)
    hs_results[hs] = acc
    RESULTS[f'T4_hs{hs}'] = acc
    print(f"  hidden={hs} -> {acc*100:.2f}%")
    print()

print(f"Hidden size results: 128={hs_results.get(128,0)*100:.2f}%, 256=59.15%, 512={hs_results.get(512,0)*100:.2f}%")


## 7. Experiment T5: Best Combination

Take the best tau, best lr, and test with 2 layers.

In [ ]:
print("=" * 60)
print("T5: BEST COMBINATION")
print("=" * 60)

# Find best tau and lr
best_tau_val = max(tau_results, key=tau_results.get)
best_lr_val = max(lr_results, key=lr_results.get)
# Also check if base lr=1e-3 was better than all swept values
if 0.5915 >= lr_results[best_lr_val]:
    best_lr_val = 1e-3

print(f"Using: tau={best_tau_val}ms, lr={best_lr_val}")

# 1-layer with best hyperparams
print("--- 1-layer best combo ---")
m1 = SNN_BNTT_LT(40, 256, 3, n_steps=T, n_hidden_layers=1,
                  tau_mem=best_tau_val, tau_syn=best_tau_val/2, threshold_init=0.1)
t1 = Trainer(m1, train_ld, val_ld, test_ld, lr=best_lr_val, device=device)
acc_best_1l = t1.train(n_epochs=80, patience=20)
RESULTS['T5_best_1L'] = acc_best_1l

# 2-layer with best hyperparams
print("--- 2-layer best combo ---")
m2 = SNN_BNTT_LT(40, 256, 3, n_steps=T, n_hidden_layers=2,
                  tau_mem=best_tau_val, tau_syn=best_tau_val/2, threshold_init=0.1)
print(f"Params: {m2.count_params():,}")
t2 = Trainer(m2, train_ld, val_ld, test_ld, lr=best_lr_val, device=device)
acc_best_2l = t2.train(n_epochs=80, patience=20)
RESULTS['T5_best_2L'] = acc_best_2l
print(f"Learned thresholds: {m2.get_thresholds()}")


## 8. Full Evaluation of Best Model

In [ ]:
@torch.no_grad()
def full_eval(model, loader, device, name):
    model.eval()
    preds, labels = [], []
    for x, y in loader:
        x = x.to(device)
        logits, _, _ = model(x)
        preds.append(logits.argmax(1).cpu())
        labels.append(y)
    preds = torch.cat(preds).numpy()
    labels = torch.cat(labels).numpy()
    acc = (preds == labels).mean()
    f1_w = f1_score(labels, preds, average='weighted')
    f1_m = f1_score(labels, preds, average='macro')
    print(f"{name}:")
    print(f"  Accuracy: {acc*100:.2f}% | F1 weighted: {f1_w*100:.2f}% | F1 macro: {f1_m*100:.2f}%")
    print(classification_report(labels, preds, target_names=['Down', 'Stationary', 'Up']))
    return acc, f1_w, f1_m

# Find best model
best_key = max(RESULTS, key=RESULTS.get)
print(f"Best model: {best_key} ({RESULTS[best_key]*100:.2f}%)")
print()

# The best model object depends on which experiment won
# Re-evaluate the models we still have in memory
print("=" * 60)
print("DETAILED EVALUATION")
print("=" * 60)

if 'T5_best_2L' in RESULTS and RESULTS.get('T5_best_2L', 0) >= RESULTS.get('T5_best_1L', 0):
    full_eval(m2, test_ld, device, "Best: 2-layer BNTT+LT (best tau + lr)")
else:
    full_eval(m1, test_ld, device, "Best: 1-layer BNTT+LT (best tau + lr)")


## 9. Results Summary

In [ ]:
print("=" * 70)
print("COMPLETE RESULTS - FI-2010 LOB (k=50, Balanced Test, 139K samples)")
print("=" * 70)
print(f"{'Model':<40} {'Test Acc':>10}")
print("-" * 52)

for name, acc in sorted(RESULTS.items(), key=lambda x: x[1], reverse=True):
    print(f"  {name:<38} {acc*100:>8.2f}%")

print("-" * 52)
print(f"  {'LSTM (V3)':38s} {'65.99':>8s}%")
print(f"  {'CNN (V3)':38s} {'65.47':>8s}%")
print(f"  {'Random':38s} {'33.30':>8s}%")
print()

best_rsnn = max(RESULTS.values())
print(f"Best RSNN: {best_rsnn*100:.2f}%")
print(f"LSTM:      65.99%")
print(f"Gap:       {(0.6599 - best_rsnn)*100:.1f}pp")
print()
print(f"Improvement over base RSNN (37.56%): +{(best_rsnn - 0.3756)*100:.1f}pp")


In [ ]:
with open('/kaggle/working/fi2010_v3b_results.json', 'w') as f:
    json.dump({k: float(v) for k, v in RESULTS.items()}, f, indent=2)
print("Saved to fi2010_v3b_results.json")
